In [13]:
# =============================================================================
# Fig1: PM2.5 concentration & attributable premature deaths (2020 vs 2060)
# Layout: 2 rows × 8 cols — every map has its own colorbar
# =============================================================================



from pathlib import Path
 
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as mcm
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes   # ← inset colorbar
from shapely.geometry import box as shapely_box
 
# ── 1. 参数配置 ───────────────────────────────────────────────────────────────
 
PM25_DIR   = Path("/Volumes/UCL/论文工作/空气污染/weighted_pm25/avg_fer_rcp")
DEATH_DIR  = Path("/Volumes/UCL/论文工作/空气污染/health_burden/air_scenarios_5")
OUTFILE    = Path("/Users/shirley/Desktop/plots_V2/Fig1.png")
SHP_PATH   = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/city_shp/shi_en.shp")
CHINA_SHP  = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国底图-中图社审过版本/中国底图/中国面.shp")
CHINA_SHP2 = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国国界线/九段线/九段线和群岛.shp")
 
SCENARIO       = "earlypeak_NZ_CL"
PM25_COL       = "weighted_pm25_mean"
DEATH_COL      = "mo_total"
SHP_CITY_FIELD = "English"
CSV_CITY_FIELD = "city"
PROJ_STR       = "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105"
 
COL_2020    = "#4db3b3"
COL_2060    = "#ca0020"
COL_DIFF_LO = "#4db3b3"
COL_DIFF_HI = "#ca0020"
 
DIVERGING_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "rw_b", [COL_DIFF_LO, "white", COL_DIFF_HI]
)
 
# ── 2. 数据读取 ───────────────────────────────────────────────────────────────
 
def read_and_prep(directory, scenario, year, col_name, new_name):
    path = directory / f"{scenario}_{year}.csv"
    if not path.exists():
        raise FileNotFoundError(f"File missing: {path}")
    df = pd.read_csv(path)
    return (
        df.rename(columns={CSV_CITY_FIELD: SHP_CITY_FIELD, col_name: "val"})
          [[SHP_CITY_FIELD, "val"]]
          .groupby(SHP_CITY_FIELD, as_index=False)
          .agg(val=("val", "mean"))
          .rename(columns={"val": new_name})
    )
 
# ── 3. 空间数据 ───────────────────────────────────────────────────────────────
 
china_border = gpd.read_file(CHINA_SHP).to_crs(PROJ_STR)
jiudanline   = gpd.read_file(CHINA_SHP2).to_crs(PROJ_STR)
city_shp     = gpd.read_file(SHP_PATH).to_crs(PROJ_STR)
 
from pyproj import Transformer
_hhy_transformer = Transformer.from_crs("EPSG:4326", PROJ_STR, always_xy=True)
_HHY_X, _HHY_Y = _hhy_transformer.transform(
    [127.5, 98.5],   # 经度
    [50.2,  25.0],   # 纬度
)

# 南海小框边界（WGS84 → 投影坐标，只算一次）
_NANHAI_BOUNDS = (
    gpd.GeoDataFrame(geometry=[shapely_box(105, 2, 122, 24)], crs="EPSG:4326")
    .to_crs(PROJ_STR)
    .total_bounds                  # (xmin, ymin, xmax, ymax)
)
 
# ── 4. 数据合并 ───────────────────────────────────────────────────────────────
 
plot_df = (
    city_shp
    .merge(read_and_prep(PM25_DIR,  SCENARIO, 2020, PM25_COL,  "pm20"), on=SHP_CITY_FIELD, how="left")
    .merge(read_and_prep(PM25_DIR,  SCENARIO, 2060, PM25_COL,  "pm60"), on=SHP_CITY_FIELD, how="left")
    .merge(read_and_prep(DEATH_DIR, SCENARIO, 2020, DEATH_COL, "d20"),  on=SHP_CITY_FIELD, how="left")
    .merge(read_and_prep(DEATH_DIR, SCENARIO, 2060, DEATH_COL, "d60"),  on=SHP_CITY_FIELD, how="left")
)
plot_df["pm_diff"] = plot_df["pm60"] - plot_df["pm20"]
plot_df["d_diff"]  = plot_df["d60"]  - plot_df["d20"]

 # 胡焕庸线东西分类：
# 用城市几何中心的 x 坐标与胡线插值 x 比较判断东西侧
# 胡线在 AEA 投影下是两点连线，对每个城市按其 y 坐标线性插值得到胡线 x
_hhy_x0, _hhy_y0 = _HHY_X[0], _HHY_Y[0]   # 黑河（北）
_hhy_x1, _hhy_y1 = _HHY_X[1], _HHY_Y[1]   # 腾冲（南）
 
def _hhy_x_at_y(y):
    """在胡焕庸线上，给定 y 坐标（AEA），返回对应的 x 坐标。"""
    t = (y - _hhy_y1) / (_hhy_y0 - _hhy_y1)   # 0=腾冲, 1=黑河
    return _hhy_x1 + t * (_hhy_x0 - _hhy_x1)
 
_cx = plot_df.geometry.centroid.x
_cy = plot_df.geometry.centroid.y
plot_df["hhy_side"] = np.where(_cx > _hhy_x_at_y(_cy), "East", "West")
 
# 调试：打印 d_diff 范围，确认是否有正负值
print(f"[debug] d_diff  range: {plot_df['d_diff'].min():.2f} ~ {plot_df['d_diff'].max():.2f}")
print(f"[debug] pm_diff range: {plot_df['pm_diff'].min():.2f} ~ {plot_df['pm_diff'].max():.2f}")
 
# ── 5. 全局样式 ───────────────────────────────────────────────────────────────
 
plt.rcParams.update({
    "font.family":      "Arial",
    "font.size":        6,
    "axes.titlesize":   7,
    "axes.titleweight": "bold",
    "axes.titlepad":    2,
})
 
# ── 6. 九段线小框函数 ─────────────────────────────────────────────────────────
 
def _add_nanhai_inset(parent_ax, col, norm, cmap):
    """在地图右下角添加九段线南海小框。"""
    x1, y1, x2, y2 = _NANHAI_BOUNDS
 
    axins = parent_ax.inset_axes([0.8, 0.01, 0.15, 0.2])   # 右下角 # (xmin, ymin, xmax, ymax)
    axins.set_facecolor("white")
 
    plot_df.plot(column=col, ax=axins, cmap=cmap, norm=norm,
                 linewidth=0, missing_kwds={"color": "lightgrey"})
    china_border.plot(ax=axins, facecolor="none", edgecolor="black", linewidth=0.2)
    jiudanline.plot(ax=axins, edgecolor="black", linewidth=0.4)
 
    axins.set_xlim(x1, x2)
    axins.set_ylim(y1, y2)
 
    # 隐藏刻度，保留边框细线
    axins.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in axins.spines.values():
        spine.set_linewidth(0.5)
        spine.set_color("black")
 
# ── 7. 地图绘图函数（inset colorbar + 九段线小框） ────────────────────────────
 
def draw_map(ax, col, title, unit="", is_diff=False):
    """
    colorbar 用 inset_axes 嵌入地图左下角，不占用 GridSpec 额外列。
    unit    : 显示在 colorbar 上方的单位字符串，如 "μg/m³"、"Δ thousands"
    is_diff : True → 红蓝对称色阶，支持正负值
    """
    vals = plot_df[col].dropna().values
 
    if is_diff:
        vmin_val = np.nanmin(vals)
        vmax_val = np.nanmax(vals)
 
        if vmin_val < 0 and vmax_val > 0:
            # ── 有正有负（如 deaths diff）：双侧发散，用实际范围 ──
            # TwoSlopeNorm 将 colormap 在 vcenter=0 处五五分，两侧各自线性映射
            # → 负值用满蓝色半段，正值用满红色半段，颜色都清晰可见
            norm = mcolors.TwoSlopeNorm(vmin=vmin_val, vcenter=0, vmax=vmax_val)
            cmap = DIVERGING_CMAP
 
        elif vmax_val <= 0:
            # ── 全负值（如 pm_diff）：蓝→白单侧色图，无红色 ──
            # 不用 TwoSlopeNorm，因为它永远五五分 colormap，
            # 哪怕 vmax 只有 0.01，红色视觉上仍占一半
            cmap = mcolors.LinearSegmentedColormap.from_list(
                "blue_white", [COL_DIFF_LO, "white"]
            )
            norm = mcolors.Normalize(vmin=vmin_val, vmax=0)
            # vmax 固定为 0，保证 white = 0（无变化）语义一致
 
        else:
            # ── 全正值：白→红单侧色图，无蓝色 ──
            cmap = mcolors.LinearSegmentedColormap.from_list(
                "white_red", ["white", COL_DIFF_HI]
            )
            norm = mcolors.Normalize(vmin=0, vmax=vmax_val)
 
        fmt = mticker.FuncFormatter(lambda x, _: f"{x:+.0f}")
    else:
        norm = mcolors.Normalize(vmin=np.nanmin(vals), vmax=np.nanmax(vals))
        cmap = mcm.magma_r
        fmt  = mticker.FuncFormatter(lambda x, _: f"{x:,.0f}")
 
    plot_df.plot(column=col, ax=ax, cmap=cmap, norm=norm,
                 linewidth=0, missing_kwds={"color": "lightgrey"})
    china_border.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.1)
    jiudanline.plot(ax=ax, edgecolor="black", linewidth=0.2)
    ax.plot(_HHY_X, _HHY_Y, color="black", linewidth=0.6,
            linestyle="--", dashes=(4, 3), zorder=5)
    ax.text(_HHY_X[0]+100000, _HHY_Y[0]+50000, "Hu line", fontsize=5, color="black", zorder=5)
    
    # 裁剪到中国边界范围，消除地图上方地理留白
    xmin, ymin, xmax, ymax = china_border.total_bounds
    pad_x = (xmax - xmin) * 0.01
    pad_y = (ymax - ymin) * 0.01
    ax.set_xlim(xmin - pad_x, xmax + pad_x)
    ax.set_ylim(ymin - pad_y, ymax + pad_y)    
       
    ax.set_title(title, loc="left")
    ax.set_axis_off()
 
    # ── inset colorbar（横向，地图左下角，约 45% 地图宽） ──
    cax = inset_axes(
        ax,
        width="45%", height="5%",          # 地图的 45% 宽、5% 高
        loc="lower left",
        bbox_to_anchor=(0.03, 0.04, 1, 1.2),  # (x0, y0, w_scale, h_scale) in axes coords
        bbox_transform=ax.transAxes,
        borderpad=0,
    )
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cb = plt.colorbar(sm, cax=cax, orientation="horizontal")
    cb.ax.xaxis.set_major_formatter(fmt)
    cb.ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=4, prune="both"))
    cb.ax.tick_params(labelsize=6, direction="in", pad=1, length=2)
    # 去掉 colorbar 黑色外框
    cb.outline.set_visible(False)
    # 单位标注在 colorbar 上方左对齐，简洁明了
    if unit:
        cax.set_title(unit, fontsize=5, loc="left", pad=1)
 
    # 九段线小框
    _add_nanhai_inset(ax, col, norm, cmap)
 
# ── 7. 分布对比图函数 ─────────────────────────────────────────────────────────
 # x 轴位置：2020组(1,2,3) 和 2060组(4.5,5.5,6.5)，中间留间距区分年份
_POSITIONS = {
    "All-2020":  1.0, "East-2020": 1.8, "West-2020": 2.6,
    "All-2060":  4.0, "East-2060": 4.8, "West-2060": 5.6,
}
_BOX_COLORS = {
    "All":  "grey",
    "East": "#4db3b3",   # 暖橙：胡线以东
    "West": "#ca0020",   # 绿灰：胡线以西
}
def draw_comparison_plot(ax, col_2020, col_2060, title, y_lab):
    """
    6 箱型图：All / East / West × 2020 / 2060。
    左侧三组 = 2020，右侧三组 = 2060，中间留空隙区分。
    """
    groups = {
        "All":  plot_df,
        "East": plot_df[plot_df["hhy_side"] == "East"],
        "West": plot_df[plot_df["hhy_side"] == "West"],
    }
 
    all_data, positions, colors = [], [], []
    for year, col in [("2020", col_2020), ("2060", col_2060)]:
        for grp_name, grp_df in groups.items():
            vals = grp_df[col].dropna().values
            all_data.append(vals)
            positions.append(_POSITIONS[f"{grp_name}-{year}"])
            colors.append(_BOX_COLORS[grp_name])
 
    rng = np.random.default_rng(42)
 
    # 抖动散点（在箱体之下绘制，zorder=2）
    for pos, arr, color in zip(positions, all_data, colors):
        jitter = rng.uniform(-0.20, 0.20, size=len(arr))
        ax.scatter(pos + jitter, arr,
                   s=1, alpha=0.30, color=color, linewidths=0, zorder=2)
 
    bp = ax.boxplot(
        all_data, positions=positions, widths=0.55,
        patch_artist=True, showcaps=True,
        flierprops=dict(marker="none"),
        medianprops=dict(color="black", linewidth=0.8),
        whiskerprops=dict(linewidth=0.4),
        capprops=dict(linewidth=0.4),
        boxprops=dict(linewidth=0.2, edgecolor="black"),
        zorder=3,
    )
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
        
    # 中位数标注（显示在箱体上方）
    for pos, arr in zip(positions, all_data):
        med = np.nanmedian(arr)*1.01  
        ax.text(pos, med, f"{med:.1f}",
                va="bottom", ha="center", fontsize=5, color="black", zorder=4)
 
    # x 轴：年份标签在各组中心
    ax.set_xticks([1.8, 4.8])
    ax.set_xticklabels(["2020", "2060"], fontsize=5)
    ax.set_xlim(0.3, 6.9)
 
    # 图例
    handles = [plt.Rectangle((0, 0), 1, 1, fc=_BOX_COLORS[g], alpha=0.75, lw=0)
               for g in ["All", "East", "West"]]
    ax.legend(handles, ["All", "East", "West"],
              fontsize=5, frameon=False, loc="upper right",
              handlelength=0.8, handletextpad=0.3, labelspacing=0.3)
 
    ax.set_title(title, loc="left")
    ax.set_ylabel("")
    ax.text(-0.25, 0.5, y_lab, fontsize=6,
            transform=ax.transAxes, rotation=90,
            va="center", ha="center")
 
    y0, y1 = ax.get_ylim()
    ax.set_ylim(y0 - (y1 - y0) * 0.03, y1 + (y1 - y0) * 0.10)
 
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.spines[["top", "right", "bottom"]].set_visible(False)
    ax.spines["left"].set_linewidth(0.3)
    ax.grid(axis="y", linewidth=0.3, color="lightgrey", zorder=0)
    ax.tick_params(axis="both", labelsize=6, length=0)
 
# ── 8. 布局：4列 GridSpec ────────────────────────────────────────────────────
#
#  col:   0       1       2       3
#         map_a   map_b   map_c   violin
#  宽：    1       1       1      0.65
 
fig = plt.figure(figsize=(18 / 2.54, 12/ 2.54), dpi=300)
 
gs = GridSpec(
    nrows=2, ncols=4,
    figure=fig,
    width_ratios=[1, 1, 1, 0.75],
    hspace=0.25,
    wspace=0.25,
)
 
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[0, 2])
ax_d = fig.add_subplot(gs[0, 3])
 
ax_e = fig.add_subplot(gs[1, 0])
ax_f = fig.add_subplot(gs[1, 1])
ax_g = fig.add_subplot(gs[1, 2])
ax_h = fig.add_subplot(gs[1, 3])
 
# ── 9. 填充各子图 ─────────────────────────────────────────────────────────────
 
draw_map(ax_a, "pm20",    r"a  ",  unit=r"PM$_{2.5}$ (μg/m$^{3}$)")
draw_map(ax_b, "pm60",    r"b  ",  unit=r"PM$_{2.5}$ (μg/m$^{3}$)")
draw_map(ax_c, "pm_diff", r"c  ",        unit=r"PM$_{2.5}$ (μg/m$^{3}$)",  is_diff=True)
draw_comparison_plot(ax_d, "pm20", "pm60", r"d ", r"PM$_{2.5}$ (μg/m$^{3}$)")
 
draw_map(ax_e, "d20",    "e  ",  unit="Deaths (K)")
draw_map(ax_f, "d60",    "f  ",  unit="Deaths (K)")
draw_map(ax_g, "d_diff", "g  ",   unit="Deaths (K)",  is_diff=True)
draw_comparison_plot(ax_h, "d20", "d60", "h  ", "Deaths (K)")
 
# ── 10. 保存 ──────────────────────────────────────────────────────────────────
 
OUTFILE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTFILE, dpi=400, bbox_inches="tight", facecolor="white")
plt.close(fig)
print(f"✓ Saved → {OUTFILE}")
 

[debug] d_diff  range: -63.45 ~ 43.37
[debug] pm_diff range: -43.97 ~ -1.83
✓ Saved → /Users/shirley/Desktop/plots_V2/Fig1.png
